# SafeVision Notebook 5: Report writer + fact checker (N5, N5b) and the full pipeline

**Goal:** join all parts into the full SafeVision pipeline and test the **written report**:

photo → detector (NB1) → worker matching (NB2) → **findings** → accident-risk classifier (NB3) + rule search (NB4) → **report writer (LLM)** → **fact checker (NLI)** → final report.

**Why a fact checker?** Paper [1] found that a large vision-language model flags many false helmet violations (precision 20.4%), and RAG studies [9], [10], [13] show that answers can still contain unsupported statements. So every sentence of the report is checked by a natural-language-inference (NLI) model against the evidence (findings, retrieved rules, risk prediction). Unsupported sentences are sent back to the LLM once to be corrected.

| ID | Question | Compared | Scores |
|---|---|---|---|
| **E8a** | Does grounding make the report more faithful? | V0 template (no LLM); V1 LLM with findings only; V2 LLM + rules + risk (RAG); V3 = V2 + fact check and rewrite | % supported / unsupported / contradicted sentences, invented citations, citation precision |
| **E8b** | Does the report say the same thing as the human expert? | same variants, helmet-violation images | ROUGE-L, BERTScore vs the ConstructionSite 10k human reason [1] |
| **E8c** | Does the fact checker catch wrong statements? | true facts vs automatically corrupted facts (wrong numbers, flipped meaning, fake rule numbers) | detection rate, false-alarm rate |
| **E8d** *(optional)* | Does a bigger LLM help? | Llama 3.1 8B vs Llama 3.3 70B (both on Groq) | as E8a |
| **E9** | Is it fast enough for the app? | each pipeline step | milliseconds per image |

**Needs:** a GPU runtime; secrets `HF_TOKEN` (artifacts + ConstructionSite 10k) and `GROQ_API_KEY` (console.groq.com). Keys are read from secrets only. **Runtime:** about 20-30 min (the Groq free tier allows 30 requests per minute).

## 0. Install libraries

In [ ]:
# [colab-only]
%pip install -q ultralytics sentence-transformers faiss-cpu rank-bm25 rouge-score bert-score datasets openpyxl huggingface_hub

## 1. Configuration (edit this cell only)

In [ ]:
# CONFIG
from pathlib import Path
import re, json, random, time

OUT = Path("/content/safevision_outputs_nb5")
ART = Path("/content/artifacts")                     # artifacts from Notebooks 1-4 are placed here
DETECTOR_WEIGHTS = ART / "detector_best.pt"
RUN_CONFIG = ART / "run_config.json"
THRESHOLDS = ART / "matching_thresholds.json"
CLASSIFIER_DIR = ART / "accident_classifier"
RULE_INDEX_DIR = ART / "rule_index"
CS10K_HUB = "LouisChen15/ConstructionSite"
CS10K_LOCAL = None

MATCH_METHOD = "box"          # 'box' or 'pose' (use the better method from Notebook 2)
IMGSZ = 960                   # overwritten by run_config.json
N_HELMET, N_CLEAN = 40, 20    # test images with a helmet violation / without a rule-1 violation
LLM_BACKEND = "groq"          # 'groq' | 'ollama' | 'hf'
LLM_MODEL = "llama-3.1-8b-instant"
LLM_MODEL_BIG = "llama-3.3-70b-versatile"   # E8d
RUN_E8D = False               # the 70B model has a smaller free daily quota; turn on if there is quota left
NLI_MODEL = "cross-encoder/nli-deberta-v3-base"
RUN_BERTSCORE = True
N_RULES = 3                   # rules given to the report writer
HELMET_WORDS = re.compile(r"\b(hard ?hats?|hardhats?|helmets?|head protection|safety hats?)\b", re.I)
SEED = 42

RES, FIGS = OUT / "results", OUT / "figures"
for p in [OUT, RES, FIGS, ART]:
    p.mkdir(parents=True, exist_ok=True)
random.seed(SEED)

## 2. Results logger

In [ ]:
# Results logger
import csv, datetime
import pandas as pd
LOG_CSV = RES / "experiments_log.csv"
LOG_FIELDS = ["experiment", "run", "variant", "llm", "dataset", "metric", "value", "notes", "timestamp"]

def log_result(experiment, run, variant, llm, metric, value, notes="", dataset="ConstructionSite10k test sample"):
    # Append one measured number to the results log.
    new_file = not LOG_CSV.exists()
    with open(LOG_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=LOG_FIELDS)
        if new_file:
            w.writeheader()
        w.writerow(dict(experiment=experiment, run=run, variant=variant, llm=llm, dataset=dataset, metric=metric,
                        value=round(float(value), 4), notes=notes, timestamp=datetime.datetime.now().isoformat(timespec="seconds")))

def read_log():
    return pd.read_csv(LOG_CSV) if LOG_CSV.exists() else pd.DataFrame(columns=LOG_FIELDS)

## 3. Secrets and artifacts from Notebooks 1-4

Downloads the detector (NB1), matching thresholds (NB2), accident classifier (NB3) and rule index (NB4) from the **private** Hugging Face repo `safevision-artifacts`. If that fails, upload the four output ZIPs (`safevision_nb1_outputs.zip` ... `nb4`) when asked.

In [ ]:
# [colab-only]
import os, zipfile, shutil
from google.colab import userdata, files
from huggingface_hub import login, snapshot_download, HfApi

token = userdata.get("HF_TOKEN"); login(token=token)
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")       # read from Secrets, never printed

def place(src_root):
    # Copy the needed files from an unpacked artifacts folder into ART.
    src_root = Path(src_root)
    for pat, dest in [("**/detector_best.pt", DETECTOR_WEIGHTS), ("**/run_config.json", RUN_CONFIG),
                      ("**/matching_thresholds.json", THRESHOLDS)]:
        hits = list(src_root.glob(pat))
        if hits and not dest.exists():
            shutil.copy(hits[0], dest)
    for name, dest in [("accident_classifier", CLASSIFIER_DIR), ("rule_index", RULE_INDEX_DIR)]:
        hits = [p for p in src_root.glob(f"**/{name}") if p.is_dir()]
        if hits and not dest.exists():
            shutil.copytree(hits[0], dest)

try:
    user = HfApi(token=token).whoami()["name"]
    place(snapshot_download(f"{user}/safevision-artifacts", token=token,
                            allow_patterns=["nb1_vision_detector/models/*", "nb1_vision_detector/run_config.json",
                                            "nb2_worker_matching/results/matching_thresholds.json",
                                            "nb3_accident_text/models/accident_classifier/**", "nb4_rule_search/models/rule_index/**"]))
except Exception as e:
    print("HF download failed:", e)

# Anything still missing (e.g. the vision files are in a different HF account)? Upload the output ZIPs.
def missing():
    return [n for n, p in [("detector_best.pt (NB1 ZIP)", DETECTOR_WEIGHTS), ("accident_classifier (NB3 ZIP)", CLASSIFIER_DIR),
                           ("rule_index (NB4 ZIP)", RULE_INDEX_DIR)] if not p.exists()]
while missing():
    print("Missing:", missing(), "-> upload the matching safevision_nbX_outputs.zip files (vision ZIPs come from the vision owner).")
    up = files.upload()
    if not up:
        break
    for name, data in up.items():
        zipfile.ZipFile(__import__("io").BytesIO(data)).extractall(ART / Path(name).stem)
    place(ART)
print("Artifacts:", sorted(p.name for p in ART.iterdir()), "| thresholds from NB2:", THRESHOLDS.exists())

## 4. Shared SafeVision modules

These four cells are the **same files the web-app backend uses** (`backend/app/*.py`), so the app behaves exactly like the evaluated system:
- `matching.py`: worker matching (Notebook 2, based on [4]),
- `classifier.py`: accident-risk classifier (Notebook 3, based on [6], [7]),
- `rules.py`: hybrid rule search (Notebook 4, based on [9], [10]),
- `report.py`: findings, prompts, LLM client and template report,
- `factcheck.py`: NLI fact checker.

In [ ]:
# ---- backend/app/matching.py ----
"""SafeVision worker matching (step V2).

Turns raw detector boxes (person, head, helmet, vest) into one record per
worker with a helmet status and a vest status: "yes", "no" or "unsure".

Two ways to find where a worker's head and torso are:
  * "box":  fixed fractions of the person box (top 30% = head area,
            20-65% = torso area). Simple, no extra model.
  * "pose": body keypoints from YOLO11-pose (nose/eyes/ears = head,
            shoulders/hips = torso), following the idea of matching PPE
            to body parts with pose estimation (Lopez et al., 2025, paper [4]).
If pose keypoints are missing for a person, "pose" falls back to "box".

The same file is used by Notebook 2 (experiments E3/E4) and by the backend,
so the app behaves exactly like the evaluated system.
"""
from __future__ import annotations

CLASSES = ["person", "head", "helmet", "vest"]

# COCO keypoint indices used by YOLO11-pose
HEAD_KPTS = [0, 1, 2, 3, 4]          # nose, left/right eye, left/right ear
TORSO_KPTS = [5, 6, 11, 12]          # left/right shoulder, left/right hip


# ----------------------------------------------------------------- geometry
def area(b):
    """Area of an [x1, y1, x2, y2] box (0 if the box is empty)."""
    return max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])


def intersection(a, b):
    """Area where two boxes overlap."""
    return area([max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])])


def iou(a, b):
    """Intersection over union of two boxes (0 = no overlap, 1 = identical)."""
    inter = intersection(a, b)
    union = area(a) + area(b) - inter
    return inter / union if union > 0 else 0.0


def inside_fraction(inner, outer):
    """Share of `inner` that lies inside `outer` (e.g. how much of a helmet is in the head area)."""
    a = area(inner)
    return intersection(inner, outer) / a if a > 0 else 0.0


# ------------------------------------------------------------ detections
def detections_from_result(result, min_conf=0.10):
    """Convert one Ultralytics detection result into {class_name: [(box, conf), ...]}."""
    dets = {c: [] for c in CLASSES}
    names = result.names
    for box, conf, cls in zip(result.boxes.xyxy.tolist(), result.boxes.conf.tolist(), result.boxes.cls.tolist()):
        name = names[int(cls)]
        if name in dets and conf >= min_conf:
            dets[name].append((box, float(conf)))
    return dets


def poses_from_result(result, kpt_conf=0.5):
    """Convert one YOLO11-pose result into a list of (person_box, {kpt_index: (x, y)})."""
    out = []
    if result.keypoints is None:
        return out
    xy = result.keypoints.xy.tolist()
    kc = result.keypoints.conf.tolist() if result.keypoints.conf is not None else None
    for i, box in enumerate(result.boxes.xyxy.tolist()):
        pts = {}
        for k, (x, y) in enumerate(xy[i]):
            if kc is None or kc[i][k] >= kpt_conf:
                pts[k] = (x, y)
        out.append((box, pts))
    return out


# ------------------------------------------------------- body regions
def box_regions(p):
    """Head and torso areas as fixed fractions of a person box (method 'box')."""
    x1, y1, x2, y2 = p
    h = y2 - y1
    head = [x1, y1 - 0.05 * h, x2, y1 + 0.30 * h]      # a little above the box too: helmets stick out
    torso = [x1, y1 + 0.20 * h, x2, y1 + 0.65 * h]
    return head, torso


def pose_regions(p, pose_list):
    """Head and torso areas from the best-matching pose skeleton (method 'pose'); None if unavailable."""
    best, best_iou = None, 0.5                          # pose person must overlap the detected person well
    for pbox, pts in pose_list:
        v = iou(p, pbox)
        if v > best_iou:
            best, best_iou = pts, v
    if best is None:
        return None
    head_pts = [best[k] for k in HEAD_KPTS if k in best]
    torso_pts = [best[k] for k in TORSO_KPTS if k in best]
    if len(head_pts) < 2 or len(torso_pts) < 2:
        return None
    ph = p[3] - p[1]
    hx = [x for x, _ in head_pts]; hy = [y for _, y in head_pts]
    cx, cy = sum(hx) / len(hx), sum(hy) / len(hy)
    s = max(max(hx) - min(hx), max(hy) - min(hy), 0.08 * ph)
    head = [cx - 1.2 * s, cy - 2.0 * s, cx + 1.2 * s, cy + 0.8 * s]     # helmets sit above the eyes
    tx = [x for x, _ in torso_pts]; ty = [y for _, y in torso_pts]
    pad = 0.1 * (max(tx) - min(tx) + 1)
    torso = [min(tx) - pad, min(ty) - pad, max(tx) + pad, max(ty) + pad]
    return head, torso


def _best(items, region, min_inside, used):
    """Highest-confidence item (index, box, conf) mostly inside `region` and not used by another worker."""
    best = None
    for i, (b, c) in enumerate(items):
        if i in used:
            continue
        if inside_fraction(b, region) >= min_inside and (best is None or c > best[2]):
            best = (i, b, c)
    return best


# ------------------------------------------------------------ main function
def match_workers(dets, img_h, pose_list=None, method="box", conf_yes=0.40, conf_no=0.40,
                  min_person_conf=0.30):
    """Assign helmets, heads and vests to workers.

    Returns a list of worker dicts:
      box, source ('person' or 'head-only'), person_conf,
      helmet ('yes'/'no'/'unsure'), helmet_conf, vest ('yes'/'no'/'unsure'), vest_conf.

    Rules (helmet):  matched helmet with conf >= conf_yes            -> 'yes'
                     else matched bare head with conf >= conf_no      -> 'no'
                     else                                             -> 'unsure'
    Rules (vest):    matched vest with conf >= conf_yes               -> 'yes'
                     else person clearly visible (conf >= 0.5 and
                          at least 15% of the image height)           -> 'no'
                     else                                             -> 'unsure'
    Bare heads that belong to no detected person (occlusion, crowds) become
    extra 'head-only' workers, so a hidden body does not hide a missing helmet.
    """
    workers = []
    used = {"head": set(), "helmet": set(), "vest": set()}
    persons = sorted([p for p in dets["person"] if p[1] >= min_person_conf], key=lambda t: -t[1])
    for pbox, pconf in persons:
        regions = pose_regions(pbox, pose_list) if (method == "pose" and pose_list) else None
        head_r, torso_r = regions if regions else box_regions(pbox)
        helmet = _best(dets["helmet"], head_r, 0.5, used["helmet"])
        head = _best(dets["head"], head_r, 0.5, used["head"])
        vest = _best(dets["vest"], torso_r, 0.3, used["vest"])
        for key, hit in (("helmet", helmet), ("head", head), ("vest", vest)):
            if hit:
                used[key].add(hit[0])
        if helmet and helmet[2] >= conf_yes:
            h_status, h_conf = "yes", helmet[2]
        elif head and head[2] >= conf_no:
            h_status, h_conf = "no", head[2]
        else:
            h_status, h_conf = "unsure", max([x[2] for x in (helmet, head) if x] or [0.0])
        visible = pconf >= 0.5 and (pbox[3] - pbox[1]) >= 0.15 * img_h
        if vest and vest[2] >= conf_yes:
            v_status, v_conf = "yes", vest[2]
        elif visible:
            v_status, v_conf = "no", pconf
        else:
            v_status, v_conf = "unsure", vest[2] if vest else 0.0
        workers.append(dict(box=pbox, source="person", person_conf=pconf, helmet=h_status,
                            helmet_conf=h_conf, vest=v_status, vest_conf=v_conf))
    # Bare heads with no person around them -> extra workers without a helmet.
    for i, (hb, hc) in enumerate(dets["head"]):
        if i in used["head"] or hc < conf_no:
            continue
        if any(iou(hb, hl) > 0.3 or inside_fraction(hl, hb) > 0.5 for hl, _ in dets["helmet"]):
            continue                                       # a helmet sits on this head
        workers.append(dict(box=hb, source="head-only", person_conf=0.0, helmet="no",
                            helmet_conf=hc, vest="unsure", vest_conf=0.0))
    return workers


def image_has_helmet_violation(workers):
    """Image-level decision: at least one worker clearly without a helmet."""
    return any(w["helmet"] == "no" for w in workers)

In [ ]:
# ---- backend/app/classifier.py ----
"""SafeVision accident-type classifier (N2): loads the context-only multi-task DistilBERT saved by Notebook 3.

Predicts the likely accident type (OIICS event group), injured body-part group and severity for a
situation description, following Wang et al. 2025 [6] (OIICS groups, DistilBERT) and Cao et al. 2026 [7].
"""
import json, re
from pathlib import Path


class AccidentClassifier:
    """Multi-task text classifier: shared encoder + one linear head per output."""

    def __init__(self, model_dir, device="cpu"):
        import torch, torch.nn as nn
        from transformers import AutoModel, AutoTokenizer
        d = Path(model_dir)
        meta = json.load(open(d / "labels.json"))
        self.targets, self.max_len, self.device = meta["targets"], meta["max_len"], device
        self.out_re = re.compile(rf"\b({meta['outcome_words']})\b", re.I)
        self.mech_re = re.compile(rf"\b({meta['mechanism_words']})\b", re.I)
        self.tok = AutoTokenizer.from_pretrained(d / "encoder")
        self.encoder = AutoModel.from_pretrained(d / "encoder").to(device).eval()
        hidden = self.encoder.config.hidden_size
        self.heads = nn.ModuleDict({t: nn.Linear(hidden, len(ls)) for t, ls in self.targets.items()})
        self.heads.load_state_dict(torch.load(d / "heads.pt", map_location=device))
        self.heads.to(device).eval()
        self._torch = torch

    def context_only(self, text):
        """Same masking as in training: remove outcome and mechanism words."""
        t = self.mech_re.sub(" ", self.out_re.sub(" ", text))
        return re.sub(r"\s+", " ", t).strip()

    def predict(self, text, top=3):
        """Top classes with probabilities for each output, e.g. {'event_group': [('Fall, slip or trip', 0.41), ...]}."""
        torch = self._torch
        enc = self.tok([self.context_only(text)], truncation=True, max_length=self.max_len, return_tensors="pt").to(self.device)
        with torch.no_grad():
            h = self.encoder(**{k: v for k, v in enc.items() if k in ("input_ids", "attention_mask")}).last_hidden_state[:, 0]
            out = {}
            for t, head in self.heads.items():
                p = head(h).softmax(-1)[0].cpu().numpy()
                order = p.argsort()[::-1][:top]
                out[t] = [(self.targets[t][i], round(float(p[i]), 3)) for i in order]
        return out

In [ ]:
# ---- backend/app/rules.py ----
"""SafeVision rule search (N4): hybrid BM25 + dense retrieval with reciprocal rank fusion and a cross-encoder re-ranker.

Loads the index written by Notebook 4 (chunks.parquet, chunks.faiss, config.json).
The same code runs in Notebook 5 and in the web app backend, so the app behaves like the evaluated system.
Design follows Guo et al. 2025 (EC3, hybrid RAG) [10] and Zhang et al. 2025 (BifrostRAG) [9];
RRF from Cormack et al. 2009.
"""
import json, re
from pathlib import Path

import numpy as np
import pandas as pd

STOP = set("a an the of to in on for and or with without is are was were be by at from this that it as any all no not".split())


def tokenize(text):
    """Lower-case word tokens without stop words (used by BM25)."""
    return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOP]


class RuleSearch:
    """Hybrid rule search over the regulation chunks built in Notebook 4."""

    def __init__(self, index_dir, device="cpu", use_reranker=True, emb_model=None, reranker=None):
        import faiss
        from rank_bm25 import BM25Okapi
        from sentence_transformers import SentenceTransformer, CrossEncoder
        d = Path(index_dir)
        self.cfg = json.load(open(d / "config.json"))
        self.chunks = pd.read_parquet(d / "chunks.parquet").reset_index(drop=True)
        self.faiss = faiss.read_index(str(d / "chunks.faiss"))
        self.bm25 = BM25Okapi([tokenize(t) for t in self.chunks.text])
        emb_name = emb_model or self.cfg["emb_model"]
        self.emb = SentenceTransformer(emb_name, device=device)
        # bge embedding models expect this instruction in front of short queries
        self.prefix = "Represent this sentence for searching relevant passages: " if "bge" in emb_name else ""
        self.reranker = CrossEncoder(reranker or self.cfg["reranker"], device=device) if use_reranker else None
        self.rrf_k = self.cfg.get("rrf_k", 60)
        self.candidates = self.cfg.get("candidates", 20)

    def _bm25(self, q, k):
        return list(np.argsort(-self.bm25.get_scores(tokenize(q)))[:k])

    def _dense(self, q, k):
        v = self.emb.encode([self.prefix + q], normalize_embeddings=True).astype("float32")
        return [int(i) for i in self.faiss.search(v, k)[1][0] if i >= 0]

    def _hybrid(self, q, k):
        # Reciprocal rank fusion: score = sum over lists of 1 / (rrf_k + rank)
        scores = {}
        for ranked in (self._bm25(q, self.candidates), self._dense(q, self.candidates)):
            for rank, idx in enumerate(ranked):
                scores[idx] = scores.get(idx, 0.0) + 1.0 / (self.rrf_k + rank + 1)
        return sorted(scores, key=scores.get, reverse=True)[:k]

    def search_ids(self, query, k=3):
        """Row numbers of the best k chunks (hybrid, then re-ranked if a re-ranker is loaded)."""
        cand = self._hybrid(query, self.candidates)
        if self.reranker is not None and cand:
            s = self.reranker.predict([(query, self.chunks.text[i]) for i in cand])
            cand = [cand[i] for i in np.argsort(-np.asarray(s))]
        return cand[:k]

    def search(self, query, k=3, unique_sections=True):
        """Top rules as dicts: section, heading, source, text. One chunk per section by default."""
        out, seen = [], set()
        for i in self.search_ids(query, k=k * 3 if unique_sections else k):
            row = self.chunks.iloc[i]
            if unique_sections and row.section in seen:
                continue
            seen.add(row.section)
            out.append({"section": str(row.section), "heading": str(row.heading), "source": str(row.source), "text": str(row.text)})
            if len(out) == k:
                break
        return out

In [ ]:
# ---- backend/app/report.py ----
"""SafeVision report writer (N5): turns vision findings + retrieved rules + accident-risk prediction into an incident report.

Backends: Groq API (Llama 3.1 8B, free tier), a local Ollama server, a local Hugging Face model, or a
rule-based template (no language model, used as the baseline). The prompt only allows facts from the
evidence and asks for section numbers in brackets, so the fact checker (factcheck.py) can verify them.
"""
import json, os, time

SYSTEM_PROMPT = (
    "You are a construction safety officer writing a short incident report from an automated site-photo check. "
    "Use ONLY the facts in FINDINGS, RULES and RISK. Do not invent numbers, people, places, dates or rules. "
    "Cite a rule only by a section number that appears in RULES, in square brackets, e.g. [1926.100]. "
    "If something is unclear in the findings, say it is unclear. Keep it under 180 words. "
    "Use exactly these headings, each on its own line: Summary:, Violations:, Applicable rules:, Likely accident risk:, Corrective actions:"
)


def plural(n, one, many):
    return f"{n} {one if n == 1 else many}"


def findings_from_workers(workers):
    """Worker list from matching.py -> (situation sentence for search/classifier, list of fact sentences, counts)."""
    n = len(workers)
    c = {"workers": n,
         "no_helmet": sum(w["helmet"] == "no" for w in workers),
         "helmet_unsure": sum(w["helmet"] == "unsure" for w in workers),
         "helmet_ok": sum(w["helmet"] == "yes" for w in workers),
         "no_vest": sum(w["vest"] == "no" for w in workers),
         "vest_ok": sum(w["vest"] == "yes" for w in workers)}
    facts = [f"The photo shows {plural(n, 'worker', 'workers')}." if n else "No workers were detected in the photo."]
    if n:
        facts.append(f"{plural(c['no_helmet'], 'worker is', 'workers are')} not wearing a hard hat."
                     if c["no_helmet"] else "No worker was found without a hard hat.")
        facts.append(f"{plural(c['helmet_ok'], 'worker is', 'workers are')} wearing a hard hat.")
        if c["helmet_unsure"]:
            facts.append(f"For {plural(c['helmet_unsure'], 'worker', 'workers')}, it is unclear whether a hard hat is worn.")
        facts.append(f"{plural(c['no_vest'], 'worker is', 'workers are')} not wearing a high-visibility vest."
                     if c["no_vest"] else "No worker was found without a high-visibility vest.")
    parts = [f"construction site with {plural(n, 'worker', 'workers')}"]
    if c["no_helmet"]:
        parts.append(f"{plural(c['no_helmet'], 'worker', 'workers')} without a hard hat (head protection)")
    if c["no_vest"]:
        parts.append(f"{plural(c['no_vest'], 'worker', 'workers')} without a high-visibility vest")
    situation = "; ".join(parts) + "."
    return situation, facts, c


def risk_sentences(prediction):
    """Classifier output -> evidence sentences about the likely accident (top class of each output)."""
    if not prediction:
        return []
    names = {"event_group": "accident type", "body_group": "injured body part", "severity": "severity"}
    return [f"In similar past OSHA reports, the most likely {names.get(t, t)} is {v[0][0]} (probability {v[0][1]:.2f})."
            for t, v in prediction.items()]


def build_messages(facts, rules=None, risk=None):
    """Chat messages for the report writer. rules=None gives the 'no rules' variant."""
    rules_txt = "\n".join(f"[{r['section']}] {r['text'][:600]}" for r in (rules or [])) or "(no rules provided)"
    user = ("FINDINGS:\n" + "\n".join(f"- {f}" for f in facts) +
            "\n\nRULES:\n" + rules_txt +
            "\n\nRISK:\n" + ("\n".join(f"- {r}" for r in risk) if risk else "(not available)") +
            "\n\nWrite the incident report now.")
    return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}]


def fix_messages(messages, report, bad_sentences):
    """Follow-up turn asking the model to remove or correct sentences the fact checker could not verify."""
    bad = "\n".join(f"- {s}" for s in bad_sentences)
    return messages + [{"role": "assistant", "content": report},
                       {"role": "user", "content": "These sentences are not supported by FINDINGS, RULES or RISK, or cite a rule "
                        "that is not in RULES:\n" + bad + "\nRewrite the whole report with the same headings. Remove or correct "
                        "these sentences and do not add new facts."}]


def template_report(facts, rules=None, risk=None):
    """Rule-based report without a language model (baseline variant)."""
    lines = ["Summary:", facts[0]]
    lines += ["Violations:"] + ([f"- {f}" for f in facts[1:]] or ["- None found."])
    lines += ["Applicable rules:"] + ([f"- [{r['section']}] {r['heading']}" for r in rules] if rules else ["- None retrieved."])
    lines += ["Likely accident risk:"] + ([f"- {r}" for r in risk] if risk else ["- Not available."])
    lines += ["Corrective actions:"]
    lines += (["- Stop work for the affected workers until the missing protective equipment is worn."]
              if any("not wearing" in f for f in facts) else ["- No action needed beyond routine site checks."])
    return "\n".join(lines)


class LLMClient:
    """Small chat client. backend: 'groq' | 'ollama' | 'hf' | 'template'."""

    def __init__(self, backend="groq", model=None, api_key=None, url=None, min_interval=2.1, device=None):
        self.backend = backend
        self.model = model or {"groq": "llama-3.1-8b-instant", "ollama": "llama3.1:8b",
                               "hf": "Qwen/Qwen2.5-1.5B-Instruct"}.get(backend)
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        self.url = url or {"groq": "https://api.groq.com/openai/v1/chat/completions",
                           "ollama": "http://localhost:11434/api/chat"}.get(backend)
        self.min_interval = min_interval          # Groq free tier: 30 requests per minute
        self._last = 0.0
        self._pipe = None
        if backend == "hf":
            from transformers import pipeline
            self._pipe = pipeline("text-generation", model=self.model, device_map="auto" if device is None else None,
                                  device=device)

    def chat(self, messages, temperature=0.2, max_tokens=450):
        """Send the messages and return the reply text."""
        if self.backend == "groq":
            import requests
            for attempt in range(5):
                wait = self.min_interval - (time.time() - self._last)
                if wait > 0:
                    time.sleep(wait)
                self._last = time.time()
                r = requests.post(self.url, headers={"Authorization": f"Bearer {self.api_key}"}, timeout=120,
                                  json={"model": self.model, "messages": messages, "temperature": temperature, "max_tokens": max_tokens})
                if r.status_code == 429:                       # rate limited: wait as told, then retry
                    time.sleep(float(r.headers.get("retry-after", 10)))
                    continue
                r.raise_for_status()
                return r.json()["choices"][0]["message"]["content"].strip()
            raise RuntimeError("Groq rate limit: too many retries")
        if self.backend == "ollama":
            import requests
            r = requests.post(self.url, timeout=300, json={"model": self.model, "messages": messages, "stream": False,
                                                           "options": {"temperature": temperature, "num_predict": max_tokens}})
            r.raise_for_status()
            return r.json()["message"]["content"].strip()
        if self.backend == "hf":
            out = self._pipe(messages, max_new_tokens=max_tokens, do_sample=temperature > 0,
                             temperature=max(temperature, 1e-3), return_full_text=False)
            gen = out[0]["generated_text"]
            return (gen[-1]["content"] if isinstance(gen, list) else gen).strip()
        raise ValueError(f"Unknown backend {self.backend}")

In [ ]:
# ---- backend/app/factcheck.py ----
"""SafeVision fact checker (N5b): checks every sentence of a generated report against the evidence with an NLI model.

A sentence is 'supported' if at least one evidence piece entails it, 'contradicted' if a finding contradicts it,
otherwise 'unsupported'. Rule citations (e.g. 1926.100) are also checked against the retrieved rules,
so invented section numbers are caught. Idea: grounding checks for RAG answers as discussed in [9], [10], [13].
"""
import re

CITE_RE = re.compile(r"\b(19\d\d\.\d+[a-z]?)\b")
HEADING_RE = re.compile(r"^\s*(?:#+\s*)?\**([A-Z][A-Za-z /&-]{2,40}):\**\s*(.*)$")


def split_sentences(report, skip_sections=("corrective actions",)):
    """Report text -> list of checkable sentences.

    Headings, bullet markers and very short lines are removed. Sentences under `skip_sections`
    are advice, not factual claims, so they are not checked.
    """
    out, section = [], ""
    for line in report.splitlines():
        line = line.strip()
        m = HEADING_RE.match(line)
        if m:                                                    # a heading, maybe followed by text on the same line
            section, line = m.group(1).strip().lower(), m.group(2).strip()
        if not line or section in skip_sections:
            continue
        line = re.sub(r"^([-*\u2022]|\d+[.)])\s*", "", line)      # drop bullet / numbering
        for s in re.split(r"(?<=[.!?])\s+", line):
            if len(s.split()) >= 4:
                out.append(s.strip())
    return out


def cited_sections(text):
    """All regulation section numbers cited in a text."""
    return sorted(set(CITE_RE.findall(text)))


class FactChecker:
    """NLI cross-encoder (default: cross-encoder/nli-deberta-v3-base) used as a sentence-level checker."""

    def __init__(self, model="cross-encoder/nli-deberta-v3-base", device="cpu", entail_min=0.5, contra_min=0.5):
        from sentence_transformers import CrossEncoder
        self.nli = CrossEncoder(model, device=device)
        names = {i: str(l).lower() for i, l in (self.nli.model.config.id2label or {}).items()}
        find = lambda key, default: next((i for i, n in names.items() if key in n), default)
        # documented label order of the cross-encoder NLI models: contradiction, entailment, neutral
        self.i_con, self.i_ent = find("contra", 0), find("entail", 1)
        self.entail_min, self.contra_min = entail_min, contra_min

    def _probs(self, pairs):
        import numpy as np
        logits = np.asarray(self.nli.predict(pairs, batch_size=32))
        if logits.ndim == 1:
            logits = logits[None, :]
        e = np.exp(logits - logits.max(axis=1, keepdims=True))
        return e / e.sum(axis=1, keepdims=True)

    def check(self, report, evidence, findings=None, allowed_sections=None):
        """Label every sentence of `report`.

        evidence: list of evidence texts (findings sentences, retrieved rule chunks, risk sentences).
        findings: subset of evidence that states hard facts from the photo (used for contradictions).
        allowed_sections: section numbers that were retrieved; other citations count as invented.
        """
        findings = findings or []
        allowed = set(allowed_sections or [])
        results = []
        for s in split_sentences(report):
            pairs = [(ev, s) for ev in evidence]
            p = self._probs(pairs) if pairs else None
            ent = float(p[:, self.i_ent].max()) if p is not None else 0.0
            best = evidence[int(p[:, self.i_ent].argmax())] if p is not None else ""
            n_f = [i for i, ev in enumerate(evidence) if ev in findings]
            con = float(p[n_f, self.i_con].max()) if (p is not None and n_f) else 0.0
            bad_cites = [c for c in cited_sections(s) if allowed and c not in allowed]
            if bad_cites:
                label = "invented citation"
            elif con >= self.contra_min and ent < self.entail_min:
                label = "contradicted"
            elif ent >= self.entail_min:
                label = "supported"
            else:
                label = "unsupported"
            results.append({"sentence": s, "label": label, "entail": round(ent, 3), "contra": round(con, 3),
                            "best_evidence": best[:200], "bad_citations": bad_cites})
        return results


def summarise(checks, report=""):
    """Share of supported / unsupported / contradicted sentences and citation precision for one report."""
    n = max(len(checks), 1)
    cites = cited_sections(report)
    bad = sorted({c for r in checks for c in r["bad_citations"]})
    return {"sentences": len(checks),
            "supported_pct": 100 * sum(r["label"] == "supported" for r in checks) / n,
            "unsupported_pct": 100 * sum(r["label"] == "unsupported" for r in checks) / n,
            "contradicted_pct": 100 * sum(r["label"] == "contradicted" for r in checks) / n,
            "citations": len(cites), "invented_citations": len(bad),
            "citation_precision": (len(cites) - len(bad)) / len(cites) if cites else float("nan")}

## 5. Load all models

If `run_config.json` and `matching_thresholds.json` exist, the image size and the tuned thresholds from Notebooks 1-2 are used.

In [ ]:
# Load all models
import torch, numpy as np
from ultralytics import YOLO
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if RUN_CONFIG.exists():
    IMGSZ = json.load(open(RUN_CONFIG))["best_imgsz"]
conf_yes, conf_no = 0.40, 0.40
if THRESHOLDS.exists():
    conf_yes, conf_no = json.load(open(THRESHOLDS))[MATCH_METHOD]
detector = YOLO(str(DETECTOR_WEIGHTS))
pose_model = YOLO("yolo11s-pose.pt") if MATCH_METHOD == "pose" else None
classifier = AccidentClassifier(CLASSIFIER_DIR, device=DEVICE) if CLASSIFIER_DIR.exists() else None
rules = RuleSearch(RULE_INDEX_DIR, device=DEVICE)
checker = FactChecker(NLI_MODEL, device=DEVICE)
llm = LLMClient(LLM_BACKEND, model=LLM_MODEL)
print(f"image size {IMGSZ} | thresholds yes={conf_yes} no={conf_no} | classifier: {classifier is not None} | LLM: {LLM_BACKEND}/{LLM_MODEL}")

## 6. Test images from ConstructionSite 10k

`N_HELMET` test images whose human reason mentions a missing hard hat and `N_CLEAN` images without any rule-1 violation (same labelling as Notebook 2). The human reason is the **reference text** for E8b.

In [ ]:
# Test images from ConstructionSite 10k
from datasets import load_dataset, load_from_disk
cs_test = (load_from_disk(CS10K_LOCAL) if CS10K_LOCAL else load_dataset(CS10K_HUB))["test"]

def rule1(rec):
    # (has_violation, reason) for rule 1 (PPE) of one record.
    v = rec["rule_1_violation"]
    if isinstance(v, str):
        try:
            v = json.loads(v)
        except Exception:
            return bool(v.strip()), v
    if not v:
        return False, ""
    return True, v.get("reason") or ""

helmet_idx, clean_idx, reasons = [], [], {}
for i in range(len(cs_test)):
    has, reason = rule1(cs_test[i])
    if has and HELMET_WORDS.search(reason):
        helmet_idx.append(i); reasons[i] = reason
    elif not has:
        clean_idx.append(i)
rng = random.Random(SEED)
sample = [(i, "helmet_violation") for i in rng.sample(helmet_idx, min(N_HELMET, len(helmet_idx)))] + \
         [(i, "no_violation") for i in rng.sample(clean_idx, min(N_CLEAN, len(clean_idx)))]
print(f"{len(helmet_idx)} helmet-violation and {len(clean_idx)} clean test images available; using {len(sample)}")

## 7. Run the pipeline once per image (vision → findings → risk → rules)

Everything except the LLM is computed once and reused by all report variants. The time of each step is recorded for E9.

In [ ]:
# Run the pipeline once per image (vision → findings → risk → rules)
cases = []
for i, label in sample:
    img = cs_test[i]["image"].convert("RGB"); t = {}
    t0 = time.time()
    det = detector.predict(img, imgsz=IMGSZ, conf=0.10, device=DEVICE, verbose=False)[0]
    poses = poses_from_result(pose_model.predict(img, imgsz=IMGSZ, conf=0.25, device=DEVICE, verbose=False)[0]) if pose_model else None
    workers = match_workers(detections_from_result(det), img.size[1], pose_list=poses, method=MATCH_METHOD, conf_yes=conf_yes, conf_no=conf_no)
    t["vision_ms"] = 1000 * (time.time() - t0)
    situation, facts, counts = findings_from_workers(workers)
    t0 = time.time(); pred = classifier.predict(situation) if classifier else None; t["classifier_ms"] = 1000 * (time.time() - t0)
    risk = risk_sentences(pred)
    t0 = time.time(); top_rules = rules.search(situation, k=N_RULES); t["rules_ms"] = 1000 * (time.time() - t0)
    evidence = facts + [r["text"] for r in top_rules] + risk
    cases.append(dict(idx=i, label=label, reason=reasons.get(i, ""), workers=workers, situation=situation, facts=facts,
                      counts=counts, risk=risk, rules=top_rules, evidence=evidence, times=t))
    print(f"\r{len(cases)}/{len(sample)} images", end="")
print("\nExample situation:", cases[0]["situation"]); print("Rules:", [r["section"] for r in cases[0]["rules"]])

## 8. Write and check the reports (variants V0-V3)

- **V0 template:** no LLM, fixed sentences (a faithful but stiff baseline).
- **V1 LLM, findings only:** no rules and no risk, so any rule it cites is from its own memory.
- **V2 LLM + rules + risk (RAG).**
- **V3 = V2 + fact check:** sentences the checker cannot verify are sent back once for correction.

Every report is checked against the **same evidence** so the variants are comparable. Sentences under *Corrective actions* are advice rather than claims, so the checker skips them.

In [ ]:
# Write and check the reports (variants V0-V3)
def write_report(case, variant, client):
    # Return (report text, checks list, number of LLM calls) for one case and one variant.
    allowed = [r["section"] for r in case["rules"]]
    if variant == "V0 template":
        rep, calls = template_report(case["facts"], case["rules"], case["risk"]), 0
    else:
        msgs = build_messages(case["facts"], None if variant == "V1 LLM findings only" else case["rules"],
                              None if variant == "V1 LLM findings only" else case["risk"])
        rep, calls = client.chat(msgs), 1
    checks = checker.check(rep, case["evidence"], findings=case["facts"], allowed_sections=allowed)
    if variant.startswith("V3"):
        bad = [c["sentence"] for c in checks if c["label"] != "supported"]
        if bad:
            rep = client.chat(fix_messages(msgs, rep, bad)); calls += 1
            checks = checker.check(rep, case["evidence"], findings=case["facts"], allowed_sections=allowed)
    return rep, checks, calls

VARIANTS = ["V0 template", "V1 LLM findings only", "V2 LLM + rules + risk", "V3 = V2 + fact check"]

def run_variants(client, llm_name, variants=VARIANTS):
    rows = []
    for v in variants:
        for n, case in enumerate(cases):
            t0 = time.time()
            try:
                rep, checks, calls = write_report(case, v, client)
            except Exception as e:                      # keep going if one call fails; the error is recorded
                rep, checks, calls = f"ERROR: {e}", [], 0
            rows.append(dict(variant=v, llm=llm_name if v != "V0 template" else "-", idx=case["idx"], label=case["label"],
                             report=rep, checks=checks, llm_calls=calls, ms=1000 * (time.time() - t0),
                             **summarise(checks, rep)))
            print(f"\r{v}: {n + 1}/{len(cases)}", end="")
        print()
    return pd.DataFrame(rows)

reports = run_variants(llm, LLM_MODEL)
print(reports.groupby("variant")[["supported_pct", "unsupported_pct", "contradicted_pct", "invented_citations"]].mean().round(1))

## 9. Experiment E8a: Faithfulness of each variant

In [ ]:
# Experiment E8a: Faithfulness of each variant
def e8a_table(df, exp="E8a"):
    # Average faithfulness scores per variant, logged to the results file.
    agg = df[~df.report.str.startswith("ERROR")].groupby(["variant", "llm"]).agg(
        reports=("report", "size"), supported_pct=("supported_pct", "mean"), unsupported_pct=("unsupported_pct", "mean"),
        contradicted_pct=("contradicted_pct", "mean"), invented_citations_per_report=("invented_citations", "mean"),
        citation_precision=("citation_precision", "mean"), llm_calls=("llm_calls", "mean"), ms_per_report=("ms", "mean")).reset_index()
    for _, r in agg.iterrows():
        for m in ["supported_pct", "unsupported_pct", "contradicted_pct", "invented_citations_per_report", "citation_precision", "ms_per_report"]:
            if pd.notna(r[m]):
                log_result(exp, f"{exp}_{r.variant.split()[0]}_{r.llm}", r.variant, r.llm, m, r[m])
    return agg

e8a = e8a_table(reports)
print(e8a.round(3).to_string(index=False))

## 10. Experiment E8b: Agreement with the human expert's reason

ConstructionSite 10k gives a one-to-two sentence human reason for every violation [1]. We compare it with the **Violations** part of each report (or the whole report if the heading is missing). ROUGE-L counts shared word sequences; BERTScore compares meaning with a language model.

In [ ]:
# Experiment E8b: Agreement with the human expert's reason
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def violations_part(rep):
    # Text between 'Violations:' and the next heading, or the whole report.
    m = re.search(r"Violations:?\**\s*(.*?)(\n\s*\**[A-Z][A-Za-z ]+:|\Z)", rep, re.S)
    return (m.group(1) if m else rep).strip()

pos = reports[(reports.label == "helmet_violation") & ~reports.report.str.startswith("ERROR")].copy()
pos["reference"] = pos.idx.map(reasons)
pos["candidate"] = pos.report.map(violations_part)
pos["rougeL_F"] = [scorer.score(r, c)["rougeL"].fmeasure for r, c in zip(pos.reference, pos.candidate)]
if RUN_BERTSCORE:
    from bert_score import score as bert_score
    _, _, F = bert_score(list(pos.candidate), list(pos.reference), lang="en", rescale_with_baseline=True, verbose=False)
    pos["bertscore_F"] = F.numpy()
e8b = pos.groupby("variant")[[c for c in ["rougeL_F", "bertscore_F"] if c in pos]].mean().reset_index()
for _, r in e8b.iterrows():
    for m in [c for c in ["rougeL_F", "bertscore_F"] if c in e8b]:
        log_result("E8b", f"E8b_{r.variant.split()[0]}", r.variant, LLM_MODEL, m, r[m], dataset="helmet-violation images")
print(e8b.round(3).to_string(index=False))

## 11. Experiment E8c: Does the fact checker catch wrong statements?

We take the true finding sentences and corrupt them automatically in three ways:
1. **wrong number** (2 workers → 4 workers),
2. **flipped meaning** ("is not wearing" ↔ "is wearing"),
3. **fake rule**: a sentence citing a section number that was not retrieved.

A good checker keeps the true sentences as *supported* (low false alarms) and flags the corrupted ones (high detection).

In [ ]:
# Experiment E8c: Does the fact checker catch wrong statements?
def corrupt_number(s):
    m = re.search(r"\b(\d+)\b", s)
    return s[:m.start()] + str(int(m.group(1)) + 2) + s[m.end():] if m else None

def corrupt_flip(s):
    if " not wearing" in s:
        return s.replace(" not wearing", " wearing", 1)
    if " wearing" in s:
        return s.replace(" wearing", " not wearing", 1)
    return None

rows = []
for case in cases:
    allowed = [r["section"] for r in case["rules"]]
    for f in case["facts"]:
        tests = [("true sentence", f), ("wrong number", corrupt_number(f)), ("flipped meaning", corrupt_flip(f))]
        for kind, s in tests:
            if s:
                c = checker.check(s, case["evidence"], findings=case["facts"], allowed_sections=allowed)
                if c:
                    rows.append({"kind": kind, "flagged": c[0]["label"] != "supported"})
    fake = f"This violates rule [1926.9{random.randint(10, 99)}]."
    c = checker.check(fake, case["evidence"], findings=case["facts"], allowed_sections=allowed)
    if c:
        rows.append({"kind": "fake rule", "flagged": c[0]["label"] != "supported"})
e8c = pd.DataFrame(rows).groupby("kind").flagged.agg(["mean", "size"]).rename(columns={"mean": "flagged_rate", "size": "sentences"}).reset_index()
for _, r in e8c.iterrows():
    log_result("E8c", f"E8c_{r.kind.replace(' ', '_')}", "fact checker", NLI_MODEL.split("/")[-1],
               "false_alarm_rate" if r.kind == "true sentence" else "detection_rate", r.flagged_rate)
print(e8c.round(3).to_string(index=False))

## 12. Experiment E8d (optional): Bigger LLM

In [ ]:
# Experiment E8d (optional): Bigger LLM
e8d = pd.DataFrame()
if RUN_E8D:
    big = LLMClient(LLM_BACKEND, model=LLM_MODEL_BIG)
    reports_big = run_variants(big, LLM_MODEL_BIG, variants=["V2 LLM + rules + risk", "V3 = V2 + fact check"])
    e8d = e8a_table(reports_big, exp="E8d")
    reports = pd.concat([reports, reports_big], ignore_index=True)
    print(e8d.round(3).to_string(index=False))
else:
    print("E8d skipped (RUN_E8D = False).")

## 13. Experiment E9: Speed of each pipeline step

In [ ]:
# Experiment E9: Speed of each pipeline step
times = pd.DataFrame([c["times"] for c in cases])
llm_ms = reports[reports.variant == "V3 = V2 + fact check"].ms.mean()
e9 = pd.DataFrame({"step": ["vision (detector + matching)", "accident classifier", "rule search", "report + fact check (V3)"],
                   "ms_per_image": [times.vision_ms.mean(), times.classifier_ms.mean(), times.rules_ms.mean(), llm_ms]})
for _, r in e9.iterrows():
    log_result("E9", "E9_latency", r.step, LLM_MODEL, "ms_per_image", r.ms_per_image, notes=DEVICE)
print(e9.round(1).to_string(index=False)); print("Total ≈", round(e9.ms_per_image.sum() / 1000, 1), "s per image on", DEVICE)

## 14. Charts and an example report

In [ ]:
# Charts and an example report
import matplotlib.pyplot as plt
plot = e8a.set_index("variant")[["supported_pct", "unsupported_pct", "contradicted_pct"]]
ax = plot.plot(kind="barh", stacked=True, figsize=(9, 3.5), color=["#2e7d32", "#f9a825", "#c62828"])
ax.set_xlabel("% of report sentences"); ax.set_title(f"E8a: faithfulness of the report ({LLM_MODEL})"); ax.invert_yaxis()
plt.tight_layout(); plt.savefig(FIGS / "fig_E8a_faithfulness.png", dpi=150); plt.show()

ex = reports[(reports.variant == "V3 = V2 + fact check") & (reports.label == "helmet_violation")].head(1)
if len(ex):
    r = ex.iloc[0]
    print("HUMAN REASON:", reasons.get(r.idx)); print("-" * 60); print(r.report); print("-" * 60)
    for c in r.checks:
        print(f"[{c['label']:>17}] {c['sentence'][:110]}")

## 15. Save everything (Excel + reports JSON + prompt) and ZIP

In [ ]:
# Save everything (Excel + reports JSON + prompt) and ZIP
import shutil
xlsx = RES / "E8_E9_report_fact_check.xlsx"
with pd.ExcelWriter(xlsx) as xw:
    e8a.to_excel(xw, sheet_name="E8a_faithfulness", index=False)
    e8b.to_excel(xw, sheet_name="E8b_vs_human", index=False)
    e8c.to_excel(xw, sheet_name="E8c_checker", index=False)
    if len(e8d): e8d.to_excel(xw, sheet_name="E8d_bigger_llm", index=False)
    e9.to_excel(xw, sheet_name="E9_latency", index=False)
    reports.drop(columns=["checks"]).to_excel(xw, sheet_name="all_reports", index=False)
    read_log().to_excel(xw, sheet_name="full_log", index=False)
reports.to_json(RES / "reports_with_checks.json", orient="records", indent=1)
json.dump({"system_prompt": SYSTEM_PROMPT, "llm_backend": LLM_BACKEND, "llm_model": LLM_MODEL, "nli_model": NLI_MODEL,
           "match_method": MATCH_METHOD, "imgsz": IMGSZ, "conf_yes": conf_yes, "conf_no": conf_no, "n_rules": N_RULES,
           "sample": {"helmet": N_HELMET, "clean": N_CLEAN}, "seed": SEED}, open(OUT / "run_config_nb5.json", "w"), indent=2)
archive = shutil.make_archive(str(OUT.parent / "safevision_nb5_outputs"), "zip", OUT)
print("Saved", xlsx, "| ZIP:", archive)

In [ ]:
# [colab-only]
files.download(archive)
try:
    api = HfApi(token=token)
    api.upload_folder(folder_path=str(OUT), repo_id=f"{api.whoami()['name']}/safevision-artifacts", path_in_repo="nb5_report_fact_check")
    print("Uploaded to the private HF repo.")
except Exception as e:
    print("HF upload skipped:", e)

## References

Papers used directly here are **in bold**.

1. **Chen, X., & Zou, Z. (2026). Are large pre-trained vision language models effective construction safety inspectors? *Data-Centric Engineering*. https://doi.org/10.1017/dce.2026.10044 (arXiv:2508.11011)** (ConstructionSite 10k test images and human reasons; GPT-4V PPE precision 20.4%, recall 76.4%)
4. **López, L., Suárez-Ramírez, J., Alemán-Flores, M., & Monzón, N. (2025). Automated PPE compliance monitoring in industrial environments using deep learning-based detection and pose estimation. *Automation in Construction*, 176, 106231.** (worker matching)
6. **Wang, J., et al. (2025). Classifying OSHA construction accident reports: Leveraging ensemble learning and lightweight large language models. *ECAM*. https://doi.org/10.1108/ECAM-08-2025-1273** (accident-type classifier)
9. **Zhang, Y., et al. (2025). BifrostRAG: Bridging dual knowledge graphs for multi-hop question answering in construction safety. *Automation in Construction* (2026). https://doi.org/10.1016/j.autcon.2026.106794**
10. **Guo, K. X., Wong, P. K.-Y., Cheng, J. C. P., Tao, X., & Leung, P.-H. (2025). Enhancing visual-LLM through prompt engineering and hybrid retrieval-augmented generation for site safety compliance checking. *EC3 2025*.**
13. **Hou, X., Walavalkar, A., & Xiao, B. (2026). A generative AI-based construction safety assistant using retrieval-augmented generation. *ISARC 2026*. https://doi.org/10.22260/ISARC2026/0297**
- He, P., Gao, J., & Chen, W. (2023). DeBERTaV3: Improving DeBERTa using ELECTRA-style pre-training with gradient-disentangled embedding sharing. *ICLR 2023*. (NLI fact-checker backbone)
- Williams, A., Nangia, N., & Bowman, S. (2018). A broad-coverage challenge corpus for sentence understanding through inference (MultiNLI). *NAACL 2018*.
- Lin, C.-Y. (2004). ROUGE: A package for automatic evaluation of summaries. *ACL Workshop on Text Summarization*.
- Zhang, T., Kishore, V., Wu, F., Weinberger, K. Q., & Artzi, Y. (2020). BERTScore: Evaluating text generation with BERT. *ICLR 2020*.
- Grattafiori, A., et al. (2024). The Llama 3 herd of models. arXiv:2407.21783.
- Lewis, P., et al. (2020). Retrieval-augmented generation for knowledge-intensive NLP tasks. *NeurIPS 33*.